In [1]:
import pandas as pd
from neo4j import GraphDatabase
import json
import pprint

# Sostituisci questi segnaposto con le tue credenziali Neo4j Aura
# Il tuo URI di Aura sarà simile a neo4j+s://xxxxxxx.databases.neo4j.io
NEO4J_URI      = "neo4j+ssc://e50dcde9.databases.neo4j.io"
NEO4J_USER     = "e50dcde9"
NEO4J_PASSWORD = "5Wq7pNsDXym3-uMbo0Ug0BwK_dkTmko-B0NdiDJAg0k"
NEO4J_DATABASE = "e50dcde9" # O il nome del tuo database se diverso

# Ora possiamo tentare la connessione
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print(f"Connesso a {NEO4J_URI}")
except Exception as e:
    print(f"Errore di connessione a Neo4j: {e}")

def run(cypher, **params):
    """Cypher → pandas DataFrame."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

def run_write(cypher, **params):
    """Run a write query and return its summary counters."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_write(
            lambda tx: tx.run(cypher, **params).consume().counters
        )

Connesso a neo4j+ssc://e50dcde9.databases.neo4j.io


In [2]:
# Crea indici su Reparto.Nome e Ospedale.Codice (idempotente: IF NOT EXISTS)
with driver.session(database=NEO4J_DATABASE) as s:
    s.run("CREATE INDEX reparto_nome IF NOT EXISTS FOR (r:Reparto) ON (r.Nome)")
    s.run("CREATE INDEX ospedale_codice IF NOT EXISTS FOR (o:Ospedale) ON (o.Codice)")
print("Indici verificati.")

Indici verificati.


In [ ]:
def strutture_con_reparti(isCritical,Reparto):
    if isCritical:
        query = '''
        MATCH (o:Ospedale)
        
        RETURN o.Nome, o.Codice, o.Coordinate, NULL AS isReparto'''

        return run(query)
    else:
        query = f'''
        MATCH (o:Ospedale)-[:HA_REPARTO]->(rep:Reparto)
        
        WHERE rep.Nome = "PRONTO SOCCORSO"
        
        RETURN o.Nome, o.Codice, o.Coordinate,
        CASE
            WHEN EXISTS {{
            MATCH (o)-[:HA_REPARTO]->(rep2:Reparto)
            wHERE rep2.Nome = "{Reparto}"
            }}
            THEN 1
            ELSE 0
        END AS isReparto'''

        return run(query)

In [23]:
df = strutture_con_reparti(isCritical = True,Reparto = "OSTETRICIA")

print(df.aggregate({"isReparto": "sum"}).iloc[0])

df.head()

0


,o.Nome,o.Codice,o.Coordinate,isReparto
0,OSPEDALE PAPA GIOVANNI XXIII,030905-00,"(9.63853045, 45.68701679)",None
1,SPEDALI CIVILI DI BRESCIA,030906-00,"(10.23255305, 45.55518065)",None
2,OSPEDALE MANZONI,030903-00,"(9.412681005, 45.85324969)",None
3,OSPEDALE CARLO POMA,030907-00,"(10.77189591, 45.14135331)",None
4,GRANDE OSPEDALE METROPOLITANO NIGUARDA,030913-00,"(9.188869233, 45.50915842)",None
